# RS-VLM Phase 3 — LoRA Instruction Tuning (Kaggle Version)

Before running:
1. **Internet:** ON (Settings panel).
2. **Accelerator:** GPU T4 x2 (or P100).
3. **Datasets:** Click Add Input -> Your Datasets -> Add your `eurosat-custom` dataset and your `rs-vlm-checkpoints` dataset.
   *Make sure you uploaded your Phase 2 checkpoint (`model_epoch009.pt`) to your checkpoints dataset!*

In [ ]:
!nvidia-smi

## 1. Setup & Install

In [ ]:
import os

if not os.path.exists('/kaggle/working/rs_vlm'):
    !git clone https://github.com/sharksurfauto-byte/rs_vlm.git /kaggle/working/rs_vlm
else:
    !git -C /kaggle/working/rs_vlm pull
    !find /kaggle/working/rs_vlm -name '*.pyc' -delete

%cd /kaggle/working/rs_vlm
print('Ready.')

## 2. Configure Paths

In [ ]:
import yaml, sys, os
sys.path.insert(0, '/kaggle/working/rs_vlm')

# IMPORTANT: Verify these match your Kaggle Input folder names!
# Check the exact paths by clicking the '>' in the Data panel on the right.
EUROSAT_DIR = '/kaggle/input/datasets/aliasgharjjawadwala/eurosat-custom'
RSICD_DIR   = '/kaggle/input/datasets/aliasgharjjawadwala/rsicd-dataset'
P2_CKPT     = '/kaggle/input/datasets/aliasgharjjawadwala/rs-vlm-checkpoints/model_epoch009.pt'
P3_CKPT_DIR = '/kaggle/working/checkpoints/phase3'

os.makedirs(P3_CKPT_DIR, exist_ok=True)

with open('/kaggle/working/rs_vlm/configs/colab_config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

cfg['data']['eurosat_root']    = EUROSAT_DIR
cfg['data']['rsicd_root']      = RSICD_DIR
cfg['data']['num_workers']     = 2  
cfg['phase3']['batch_size']    = 4
cfg['phase3']['epochs']        = 10
cfg['phase3']['checkpoint_dir']= P3_CKPT_DIR

with open('/kaggle/working/rs_vlm/configs/colab_config.yaml', 'w') as f:
    yaml.dump(cfg, f)

print("Config Patched for Phase 3.")

## 3. Run Phase 3 Training

In [ ]:
from training.phase3_lora import train_phase3

model = train_phase3(
    config_path='configs/colab_config.yaml',
    phase2_checkpoint=P2_CKPT,
    resume_from=None,
)